#### GraphFlow

In [1]:
import asyncio
from autogen_agentchat.agents import AssistantAgent,UserProxyAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.conditions import TextMentionTermination
from dotenv import load_dotenv
import os

# Load API key
load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

# Model client
model_client = OpenAIChatCompletionClient(model='gpt-4o', api_key=api_key)

In [2]:
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow

#### Sequential Flow

In [3]:
writer = AssistantAgent(
    name="Writer",
    description="A writer agent that generates text based on user input.",
    model_client=model_client,
    system_message="You are a creative writer. Please write a story based on the user's input.",
)

reviewer = AssistantAgent(
    name="Reviewer",
    description="A reviewer agent that provides feedback on the text generated by the writer.",
    model_client=model_client,
    system_message="You are a reviewer. Please provide feedback on the text generated by the writer.",
)

In [4]:
builder = DiGraphBuilder()
builder.add_node(writer).add_node(reviewer)
builder.add_edge(writer, reviewer)

graph = builder.build()

In [5]:
graph

DiGraph(nodes={'Writer': DiGraphNode(name='Writer', edges=[DiGraphEdge(target='Reviewer', condition=None, condition_function=None, activation_group='Reviewer', activation_condition='all')], activation='all'), 'Reviewer': DiGraphNode(name='Reviewer', edges=[], activation='all')}, default_start_node=None)

DiGraph(nodes={'Writer': DiGraphNode(name='Writer', edges=[DiGraphEdge(target='Reviewer', condition=None, condition_function=None, activation_group='Reviewer', activation_condition='all')], activation='all'), 'Reviewer': DiGraphNode(name='Reviewer', edges=[], activation='all')}, default_start_node=None)

In [ ]:
team = GraphFlow([writer,reviewer], graph)

In [7]:
stream = team.run_stream(task ="Write a good poem about India in less than 30 words.")

async for event in stream:
    print(event)

id='274f3255-e63b-415a-8474-c714ff5b2957' source='user' models_usage=None metadata={} created_at=datetime.datetime(2026, 5, 12, 13, 24, 16, 166733, tzinfo=datetime.timezone.utc) content='Write a good poem about India in less than 30 words.' type='TextMessage'
id='eccd54b5-6723-4c23-8a8a-ab8706d692a8' source='Writer' models_usage=RequestUsage(prompt_tokens=41, completion_tokens=34) metadata={} created_at=datetime.datetime(2026, 5, 12, 13, 24, 18, 995539, tzinfo=datetime.timezone.utc) content="Land of colors, vibrant and vast,  \nSacred rivers, a heritage cast.  \nUnity in diversity's sing,  \nIn India's heart, hope does spring.  " type='TextMessage'
id='79b4a287-2d85-4cff-b337-a97f84d3b0ba' source='Reviewer' models_usage=RequestUsage(prompt_tokens=81, completion_tokens=96) metadata={} created_at=datetime.datetime(2026, 5, 12, 13, 24, 20, 972368, tzinfo=datetime.timezone.utc) content='Your poem beautifully captures the essence of India in just a few words. The imagery of "colors, vibrant

In [ ]:
source='user' models_usage=None metadata={} content='Write a good poem about India in less than 30 words.' type='TextMessage'
source='Writer' models_usage=RequestUsage(prompt_tokens=409, completion_tokens=39) metadata={} content='India, a vibrant tapestry spun,  \nWith rivers, deserts, land of sun.  \nCulture diverse, festivals bright,  \nUnity in every light.  \nAncient tales in modern flight.  ' type='TextMessage'
source='Reviewer' models_usage=RequestUsage(prompt_tokens=791, completion_tokens=233) metadata={} content='This concise poem effectively captures the essence of India using limited words. It manages to convey the country\'s natural beauty, cultural diversity, and the blend of ancient and modern elements.\n\nStrengths:\n1. **Economy of Language**: You\'ve managed to encapsulate the complexity of India in a few words, maintaining a clear and coherent message.\n2. **Imagery**: The mention of "rivers, deserts, land of sun" immediately sparks vivid imagery, grounding the reader in India\'s diverse landscapes.\n3. **Cultural Highlight**: Phrases like "Culture diverse, festivals bright" succinctly highlight the rich cultural tapestry of the country.\n4. **Unity Theme**: The line "Unity in every light" effectively underscores the theme of unity amidst diversity.\n\nSuggestions for Improvement:\n1. **Rhythm and Flow**: Consider adjusting the last line slightly to enhance the rhythmic flow or provide a stronger ending. An alternative could be something like "Ancient tales, modern heights."\n\nOverall, this short poem successfully conveys the richness of India’s landscape and culture, giving readers a brief yet powerful snapshot. With minor tweaks, it can become even more resonant.' type='TextMessage'
source='DiGraphStopAgent' models_usage=None metadata={} content='Digraph execution is complete' type='StopMessage'




messages=[TextMessage(source='user', models_usage=None, metadata={}, content='Write a good poem about India in less than 30 words.', type='TextMessage'), TextMessage(source='Writer', models_usage=RequestUsage(prompt_tokens=409, completion_tokens=39), metadata={}, content='India, a vibrant tapestry spun,  \nWith rivers, deserts, land of sun.  \nCulture diverse, festivals bright,  \nUnity in every light.  \nAncient tales in modern flight.  ', type='TextMessage'), TextMessage(source='Reviewer', models_usage=RequestUsage(prompt_tokens=791, completion_tokens=233), metadata={}, content='This concise poem effectively captures the essence of India using limited words. It manages to convey the country\'s natural beauty, cultural diversity, and the blend of ancient and modern elements.\n\nStrengths:\n1. **Economy of Language**: You\'ve managed to encapsulate the complexity of India in a few words, maintaining a clear and coherent message.\n2. **Imagery**: The mention of "rivers, deserts, land of sun" immediately sparks vivid imagery, grounding the reader in India\'s diverse landscapes.\n3. **Cultural Highlight**: Phrases like "Culture diverse, festivals bright" succinctly highlight the rich cultural tapestry of the country.\n4. **Unity Theme**: The line "Unity in every light" effectively underscores the theme of unity amidst diversity.\n\nSuggestions for Improvement:\n1. **Rhythm and Flow**: Consider adjusting the last line slightly to enhance the rhythmic flow or provide a stronger ending. An alternative could be something like "Ancient tales, modern heights."\n\nOverall, this short poem successfully conveys the richness of India’s landscape and culture, giving readers a brief yet powerful snapshot. With minor tweaks, it can become even more resonant.', type='TextMessage'), StopMessage(source='DiGraphStopAgent', models_usage=None, metadata={}, content='Digraph execution is complete', type='StopMessage')] stop_reason='Stop message received'

In [12]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(model="gpt-4.1-nano")
agent_a = AssistantAgent("A", model_client=model_client, system_message="You are a helpful assistant.")
agent_b = AssistantAgent("B", model_client=model_client, system_message="Translate input to Chinese.")
agent_c = AssistantAgent("C", model_client=model_client, system_message="Translate input to Japanese.")

# Fan-out: A -> (B, C)
builder = DiGraphBuilder()
builder.add_node(agent_a).add_node(agent_b).add_node(agent_c)
builder.add_edge(agent_a, agent_b).add_edge(agent_a, agent_c)
graph = builder.build()

team = GraphFlow(
    participants=[agent_a, agent_b, agent_c],
    graph=graph,
    termination_condition=MaxMessageTermination(5),
)

async for event in team.run_stream(task="Write a short story about a cat."):
    print(event)


id='0e2b8e45-dee3-4246-88d6-146e225bf2cd' source='user' models_usage=None metadata={} created_at=datetime.datetime(2026, 5, 12, 13, 37, 42, 523206, tzinfo=datetime.timezone.utc) content='Write a short story about a cat.' type='TextMessage'
id='f0235f64-e55f-44b3-8910-a0c8b8c81c1f' source='A' models_usage=RequestUsage(prompt_tokens=26, completion_tokens=230) metadata={} created_at=datetime.datetime(2026, 5, 12, 13, 37, 46, 179241, tzinfo=datetime.timezone.utc) content='Once upon a time, in a cozy little village, there was a curious cat named Whiskers. With fur as soft as clouds and eyes that sparkled like emeralds, Whiskers loved exploring every nook and cranny of the neighborhood. Each morning, he would venture out beyond his cozy home to chase sunlight rays, hunt for elusive butterflies, and climb the tallest oak trees.\n\nOne cloudy afternoon, Whiskers discovered a hidden alleyway he’d never seen before. Intrigued, he squeezed through a creaky gate and found himself in a secret garde

In [9]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow
from autogen_ext.models.openai import OpenAIChatCompletionClient


model_client = OpenAIChatCompletionClient(model="gpt-4.1-nano")
agent_a = AssistantAgent("A", model_client=model_client, system_message="You are a helpful assistant.")
agent_b = AssistantAgent("B", model_client=model_client, system_message="Translate input to Chinese.")
agent_c = AssistantAgent("C", model_client=model_client, system_message="Translate input to Japanese.")
agent_d = AssistantAgent("D", model_client=model_client, system_message="Summarize.")


# Create a directed graph with fan-out flow A -> (B, C).
builder = DiGraphBuilder()
builder.add_node(agent_a).add_node(agent_b).add_node(agent_c).add_node(agent_d)
builder.add_edge(agent_a, agent_b).add_edge(agent_a, agent_c)
builder.add_edge(agent_b, agent_d).add_edge(agent_c, agent_d)
graph = builder.build()

# Create a GraphFlow team with the directed graph.
team = GraphFlow(
    participants=[agent_a, agent_b, agent_c, agent_d],
    graph=graph,
    termination_condition=MaxMessageTermination(5),
)

# Run the team and print the events.
async for event in team.run_stream(task="Write a short story about a cat."):
    print(event)

id='ef2b23e3-07dc-4141-a6f0-b6a4a6dcaf42' source='user' models_usage=None metadata={} created_at=datetime.datetime(2026, 5, 12, 13, 33, 32, 453156, tzinfo=datetime.timezone.utc) content='Write a short story about a cat.' type='TextMessage'
id='5e5df5a7-fa4a-465f-af97-ee553c652338' source='A' models_usage=RequestUsage(prompt_tokens=26, completion_tokens=235) metadata={} created_at=datetime.datetime(2026, 5, 12, 13, 33, 38, 269619, tzinfo=datetime.timezone.utc) content='Once upon a time in a small village, there was a curious cat named Whiskers. With sleek black fur and bright green eyes, he loved to explore every nook and cranny of the neighborhood. Every morning, Whiskers would sit on the fence, watching the sunrise and dreaming of adventures.\n\nOne day, while wandering through a garden, he discovered a tiny door hidden among the bushes. Intrigued, Whiskers nudged it open with his nose and found a secret passage leading to a magical world filled with colorful flowers, singing birds, a

In [11]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(model="gpt-4.1-nano")
agent_a = AssistantAgent(
    "A",
    model_client=model_client,
    system_message="Detect if the input is in Chinese. If it is, say 'yes', else say 'no', and nothing else.",
)
agent_b = AssistantAgent("B", model_client=model_client, system_message="Translate input to English.")
agent_c = AssistantAgent("C", model_client=model_client, system_message="Translate input to Chinese.")

# Conditional: A -> B if "yes", else A -> C
builder = DiGraphBuilder()
builder.add_node(agent_a).add_node(agent_b).add_node(agent_c)
builder.add_edge(agent_a, agent_b, condition=lambda msg: "yes" in msg.to_model_text())
builder.add_edge(agent_a, agent_c, condition=lambda msg: "yes" not in msg.to_model_text())
graph = builder.build()

team = GraphFlow(
    participants=[agent_a, agent_b, agent_c],
    graph=graph,
    termination_condition=MaxMessageTermination(5),
)

async for event in team.run_stream(task="AutoGen is a framework for building AI agents."):
    print(event)


id='75bfe173-4807-4ffd-96c2-b1627da43b87' source='user' models_usage=None metadata={} created_at=datetime.datetime(2026, 5, 12, 13, 35, 40, 422987, tzinfo=datetime.timezone.utc) content='AutoGen is a framework for building AI agents.' type='TextMessage'
id='d6503ea1-7abc-4bf5-822a-d104f1820147' source='A' models_usage=RequestUsage(prompt_tokens=47, completion_tokens=1) metadata={} created_at=datetime.datetime(2026, 5, 12, 13, 35, 41, 94036, tzinfo=datetime.timezone.utc) content='no' type='TextMessage'
id='09cbe80d-b829-4ab7-b619-18e8fd024ac6' source='C' models_usage=RequestUsage(prompt_tokens=33, completion_tokens=14) metadata={} created_at=datetime.datetime(2026, 5, 12, 13, 35, 41, 647853, tzinfo=datetime.timezone.utc) content='AutoGen 是一个用于构建人工智能代理的框架。' type='TextMessage'
messages=[TextMessage(id='75bfe173-4807-4ffd-96c2-b1627da43b87', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 5, 12, 13, 35, 40, 422987, tzinfo=datetime.timezone.utc), content='A